## 3_download_google_25D
### This notebook downloads all tiles for the area of interest. To find the list of tiles covering the area of interest, please use [this colab](https://colab.research.google.com/github/google-research/google-research/blob/master/building_detection/open_buildings_temporal_download_region_geotiffs.ipynb) mentioned in the Download section of the [main page](https://sites.research.google/gr/open-buildings/temporal/). Copy the list provided by this colab next to a script, its location is specified in the "Initial configuration" part.
### To remove connection to IBM COS save the boundary of the area of interest on a local computer and 
 - Ignore the 3rd cell starting with "cos_client = ibm_boto3.client"
 - Remove calling "upload_to_cos" in the main part of the downloading process

### Initial configuration
#### To start working with this particular notebook, you need to provide necessary credential and settings
#### Below is an template of configuration, which is necessary prepare aside of this notebook and copy & paste all content in triple quotes to the next cell's input field
#### Please make sure OSM provides boundary polygon for the specified country name!
    """
    {
    "URL_FILE": "mylist.txt",
    "FILE_PREFIX": "Gujarat",
    "REGION": "Gujarat"

    "COS_ENDPOINT_URL": "s3.private.eu-de.cloud-object-storage.appdomain.cloud",
    "COS_AUTH_ENDPOINT_URL": "https://iam.cloud.ibm.com/oidc/token",
    "COS_APIKEY": "xxx",
    "BUCKET_TIFF": "google-2.5d",
    "VIDA_COUNTRIES_BUILDINGS": "vida-countries-buildings",
    }
    """


In [6]:
# Read notebook configuration
import getpass
import json

config_str = getpass.getpass('Enter your prepared config: ')
config = json.loads(config_str)


In [ ]:
# Import necessary libraries
import rasterio
import matplotlib.pyplot as plt
from PIL import Image
import zipfile
import os
import ibm_boto3
from botocore.client import Config
import boto3
import requests
from botocore.exceptions import NoCredentialsError
import os

In [ ]:
cos_client = ibm_boto3.client(service_name='s3',
                                  ibm_api_key_id=config["COS_API_KEY_ID"],
                                  ibm_auth_endpoint=config["COS_AUTH_ENDPOINT_URL"],
                                  config=Config(signature_version='oauth'),
                                  endpoint_url=config["COS_ENDPOINT_URL"])

In [ ]:
# Download GeoTiff files using a list of downloadable urls list generated by https://colab.research.google.com/github/google-research/google-research/blob/master/building_detection/open_buildings_temporal_download_region_geotiffs.ipynb
# To get polygon WKT format use https://polygons.openstreetmap.fr/?id=1950801
# Change URL format: https://storage.googleapis.com/open-buildings-temporal-data/v1/geotiffs/39f8c_2023_06_30/tile_yQh4ZZb6AMo.tif
# Place the URL list into bucket
# Parameters - update these accordingly at config above!
file_prefix = config["FILE_PREFIX"]
bucket_name = config["BUCKET_TIFF"]
text_file_path = config["URL_FILE"]
download_directory = "/tmp/geotiffs"  # Temporary local directory for downloads

# Step 1: Download GeoTIFF files
def download_geotiff(url, output_path):
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()  # Raise exception for HTTP errors
        with open(output_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Downloaded: {output_path}")
    except Exception as e:
        print(f"Error downloading {url}: {e}")

# Step 2: Upload GeoTIFF to COS
def upload_to_cos(local_path, bucket, object_key):
    try:
        cos_client.upload_file(local_path, bucket, object_key)
        print(f"Uploaded {local_path} to COS as {object_key}")
    except NoCredentialsError:
        print("Credentials not available.")
    except Exception as e:
        print(f"Error uploading {local_path} to COS: {e}")

# Step 3: Main process
def process_geotiff_files():
   
    # Create a temporary local directory for downloads
    os.makedirs(download_directory, exist_ok=True)
    
    with open(text_file_path,'r') as url_list:
        for url in url_list:
            filename = url.split('/')[-1]  # Extract the filename from URL
            local_path = os.path.join(download_directory, filename)
            #object_key = f"file_prefix{filename}"  # Path in the COS bucket
            object_key = f"{file_prefix}_{filename}"  # Path in the COS bucket
        
            # Download the file
            download_geotiff(url, local_path)
        
            # Upload the file to COS
            upload_to_cos(local_path, bucket_name, object_key)
        
            # Cleanup local file
            os.remove(local_path)

# Run the script
if __name__ == "__main__":
    process_geotiff_files()
